# PoliMillionaire - BM25 multi-index + BERT reranker (Colab, No LLM)

Notebook Colab per usare BM25 su SimpleWiki + KELM e reranking BERT senza generazione.

Prima di eseguirlo in Colab, carica o monta una cartella che contenga almeno:

- `api_client/NLP_assignment_api_client/`
- `project/src/`
- `data/indexes/simplewiki_160w_bm25.joblib`
- `data/indexes/kelm_500k_bm25.joblib`


In [ ]:
# Se necessario, abilita GPU in Colab: Runtime > Change runtime type > T4 GPU.
!pip -q install sentence-transformers bm25s joblib numpy scikit-learn sympy requests


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

# Cambia questo path se hai copiato la repo in un'altra cartella Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/NLP_polimi_26')

CLIENT_PARENT = PROJECT_ROOT / 'api_client' / 'NLP_assignment_api_client'
SRC_DIR = PROJECT_ROOT / 'src'

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print('Project root:', PROJECT_ROOT)
print('Client path exists:', CLIENT_PARENT.exists())
print('Source path exists:', SRC_DIR.exists())


In [ ]:
import numpy as np
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import (
    load_retrieval_index,
    retrieve,
    compact_snippet,
    RetrievalDecision,
    run_all_competitions,
    summarize,
    write_logs,
)

API_URL = os.getenv('POLIMILLIONAIRE_API_URL', 'http://131.175.15.22:51111/')
USERNAME = os.getenv('POLIMILLIONAIRE_USERNAME') or input('Username: ').strip()
PASSWORD = os.getenv('POLIMILLIONAIRE_PASSWORD') or getpass('Password: ')

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} ({user.role})')


In [ ]:
WIKI_INDEX_PATH = PROJECT_ROOT / 'indexes' / 'simplewiki_160w_bm25.joblib'
KELM_INDEX_PATH = PROJECT_ROOT / 'indexes' / 'kelm_500k_bm25.joblib'

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
kelm_index = load_retrieval_index(KELM_INDEX_PATH)
indexes = [wiki_index, kelm_index]

print('Loaded SimpleWiki:', len(wiki_index['docs']))
print('Loaded KELM:', len(kelm_index['docs']))


In [ ]:
import re
import torch
from sentence_transformers import CrossEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('BERT device:', DEVICE)

bert_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2', max_length=512, device=DEVICE)

LOCAL_STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'in', 'is', 'it',
    'of', 'on', 'or', 'the', 'to', 'was', 'were', 'with', 'what', 'which', 'who',
    'when', 'where'
}

def significant_terms(text):
    return [
        t for t in re.findall(r'[a-z0-9][a-z0-9_+-]*', str(text).lower())
        if t not in LOCAL_STOPWORDS and len(t) > 2
    ]

def passage_for_bert(doc, query, max_chars=1400):
    title = str(doc.get('title') or '')
    text = re.sub(r'\s+', ' ', str(doc.get('text') or '')).strip()
    if len(text) <= max_chars:
        return f'{title}: {text}' if title else text

    lowered = text.lower()
    positions = [lowered.find(term) for term in significant_terms(query)]
    positions = [pos for pos in positions if pos >= 0]
    center = min(positions) if positions else 0
    start = max(0, center - max_chars // 3)
    end = min(len(text), start + max_chars)
    passage = text[start:end]
    return f'{title}: {passage}' if title else passage

def choose_con_bert(question, index, top_k_bm25=4):
    global_query = ' '.join([str(question.text), *[str(opt.text) for opt in question.options]])
    global_results = retrieve(index, global_query, top_k=top_k_bm25)

    option_scores = []
    for opt in question.options:
        option_query = f'{question.text} {opt.text}'
        candidates = retrieve(index, option_query, top_k=top_k_bm25)

        if candidates:
            pairs = [(option_query, passage_for_bert(doc, option_query)) for _, doc in candidates]
            bert_scores = [
                float(score)
                for score in bert_model.predict(pairs, batch_size=16, show_progress_bar=False)
            ]
            ranked = sorted(zip(bert_scores, candidates), key=lambda row: row[0], reverse=True)
            top_scores = [score for score, _ in ranked[:3]]
            bert_score = top_scores[0] + 0.10 * float(np.mean(top_scores))
            best_doc = ranked[0][1][1]
            best_bm25_score = ranked[0][1][0]
        else:
            bert_score = float('-inf')
            best_doc = None
            best_bm25_score = 0.0

        option_scores.append({
            'option_id': opt.id,
            'option_text': str(opt.text),
            'score': bert_score,
            'evidence_score': bert_score,
            'best_bm25_score': best_bm25_score,
            'top_evidence': compact_snippet(best_doc) if best_doc else None,
        })

    option_scores.sort(key=lambda row: row['score'], reverse=True)
    best = option_scores[0]
    second_score = option_scores[1]['score'] if len(option_scores) > 1 else 0.0

    return RetrievalDecision(
        option_id=int(best['option_id']),
        option_text=str(best['option_text']),
        confidence=float(best['score'] - second_score),
        option_scores=option_scores,
        evidence=[{'score': sc, **compact_snippet(dc)} for sc, dc in global_results],
    )


In [ ]:
# Smoke test locale: non chiama l'API, controlla solo retrieval + BERT.
class Obj:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

question = Obj(
    text='Who co-founded Apple Inc.?',
    options=[
        Obj(id=0, text='Steve Jobs'),
        Obj(id=1, text='Bill Gates'),
        Obj(id=2, text='Albert Einstein'),
        Obj(id=3, text='Napoleon Bonaparte'),
    ],
)

decision = choose_con_bert(question, indexes, top_k_bm25=4)
print(decision.option_text)
print([(row['option_text'], round(row['score'], 3)) for row in decision.option_scores])


In [ ]:
# Parti con 1 tentativo per verificare tempi/output. Poi aumenta.
ATTEMPTS_PER_COMPETITION = 1
STRATEGY_NAME = 'bm25_mixed_plus_bert_reranker_colab'

rows = run_all_competitions(
    client=client,
    index=indexes,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
    choose_fn=lambda question, index: choose_con_bert(question, index, top_k_bm25=4),
)

output_path = PROJECT_ROOT / 'logs' / 'simplewiki_kelm_bert_no_llm_all_competitions_colab.csv'
write_logs(rows, output_path)
print(f'Wrote {len(rows)} rows to {output_path}')


In [ ]:
summarize(rows)
